# Fraus - Fine-tune do Laya para emocao e ironia, comparado ao BERTimbau

O Laya **nao e um JEPA**: e um encoder BERT bidirecional (mmBERT-base, 322 M
de parametros no checkpoint multilingue) com uma cabeca que pontua cada opcao
de uma pergunta `choice`. O teste que ja existia no Fraus
(`fraus/sinais/ironia_laya.py`) usa esse checkpoint **sem treino nenhum**. Este
notebook ensina o MESMO checkpoint a responder duas perguntas, e so elas:

- **emocao** -- as sete classes de `NOMES_EMOCOES` (o desprezo continua
  derivado de raiva x nojo, como no notebook 03);
- **ironia** -- a mesma pergunta `PERGUNTA_IRONIA` que a API ja serve.

Um checkpoint, duas perguntas: em producao isso e UM modelo no lugar de dois.

## O que este notebook NAO promete

**Nao existe 100% de acerto honesto nestas tarefas.** Os proprios anotadores
humanos discordam em emocao e em ironia, o corpus de emocao e traducao
automatica, e a invariante 10 do projeto ja pagou por isso uma vez: o primeiro
fusor marcou 99,3% lendo um atalho do corpus. Acuracia perto de 100% aqui e
sintoma de vazamento, nao vitoria. A pergunta que o notebook responde e outra:
**o Laya treinado ganha do BERTimbau no mesmo teste, com diferenca que
sobrevive a um teste estatistico?**

Por isso a comparacao e PAREADA (os dois modelos leem exatamente os mesmos
exemplos de teste), com McNemar exato e intervalo de confianca por bootstrap,
e tem dois conjuntos que nenhum dos dois viu no treino: o XED-pt (emocao, sem
traducao automatica) e a regua de dominio de `fraus/avaliacao_ironia.py`.

## Ordem

1. ambiente, 2. pacote `fraus`, 3. script oficial e checkpoint base,
4. corpus de emocao, 5. corpus de ironia, 6. JSONL de treino, 7. treino,
8. comparacao de emocao, 9. comparacao de ironia, 10. encolhimento para o
deploy (poda de vocabulario + ONNX + INT8), 11. metricas e ZIP, 12. laudo
de comparacao para a dashboard.

Requer **GPU** (T4 basta). Rode primeiro com `MODO_FUMACA = True` (celula 7):
ele treina poucos minutos, prova que o caminho inteiro funciona e imprime a
estimativa de tempo do treino completo.

In [ ]:
# 1. GPU conferida; Drive montado se der.
!pip -q install 'laya==0.3.21' transformers datasets accelerate scikit-learn safetensors huggingface_hub

import os

import torch

assert torch.cuda.is_available(), (
    'Runtime sem GPU. Ambiente de execucao > Alterar tipo de ambiente de execucao > GPU.'
)
print('gpu:', torch.cuda.get_device_name(0))

# O Drive guarda o checkpoint treinado e e de onde vem o BERTimbau dos
# notebooks 03 e 04. Sem ele o treino roda, mas a comparacao nao tem contra
# quem comparar -- a celula 8 para com erro dizendo o que falta.
DRIVE = '/content/drive/MyDrive/fraus'
try:
    from google.colab import drive

    drive.mount('/content/drive', force_remount=True)
    RAIZ = DRIVE
    print('artefatos no Drive: sobrevivem a queda de sessao.')
except Exception as erro:
    RAIZ = '/content/fraus-local'
    print(f'AVISO: Drive indisponivel ({type(erro).__name__}: {erro}).')
    print(f'Gravando em {RAIZ} -- se a sessao cair, o treino recomeca do zero.')

DIR_TRABALHO = '/content/laya-treino'
DIR_SAIDA = f'{RAIZ}/modelos/laya-fraus'
DIR_BERT_EMOCAO = f'{RAIZ}/modelos/bertimbau-emocao'
DIR_BERT_IRONIA = f'{RAIZ}/modelos/bertimbau-ironia'
DIR_MODELOS = f'{RAIZ}/modelos'
for pasta in (DIR_TRABALHO, DIR_SAIDA):
    os.makedirs(pasta, exist_ok=True)

## 2. O pacote `fraus`

Mesma celula dos notebooks 03 e 04. As duas perguntas, o formato de cada linha
de treino e as estatisticas da comparacao **nao sao digitadas aqui**: vem de
`fraus.treino_laya` e `fraus.comparacao_modelos`, que tem teste. Treinar numa
redacao da pergunta e servir outra e treinar o modelo errado sem erro nenhum.

In [ ]:
# 2. Pacote fraus instalado a partir do repositorio.
DONO = 'Joao-Tavares0312'
NOME_REPO = 'Fraus'
# O RAMO DO TREINO. Era 'feat/motor-e-dashboard', que em 24/08/2026 estava
# 141 commits atras do main e zero a frente -- absorvido e obsoleto.
#
# O ESTRAGO era silencioso, que e o pior tipo: naquele ramo
# `montar_features` ainda aceitava UM classificador e devolvia 16 features.
# O notebook rodava sem erro, treinava feliz, e cuspia um fusor de 16
# features num repositorio cujo `NOMES_FEATURES` tem 35. O treino nao
# falhava: ele acertava o alvo errado, e so aparecia na API, no boot, como
# 'X has 35 features, but StandardScaler is expecting 16'.
#
# Se um dia precisar treinar de um ramo de trabalho, troque aqui -- mas
# confira antes que o contrato de features daquele ramo e o do destino.
RAMO = 'feat/treino-laya'  # volte para 'main' depois do merge: os modulos fraus.treino_laya e fraus.comparacao_modelos precisam existir no ramo clonado
DIR_REPO = '/content/fraus-src'

import shutil
import subprocess
import sys

# Os TRES motivos de nao haver token sao distintos, e o remedio de cada um e
# outro. `except Exception: token = None` tratava os tres como o mesmo e caia
# para um clone anonimo -- que num repo privado morre la na frente com
# "could not read Username for 'https://github.com'", mensagem que nao
# menciona Secret nenhum e manda procurar no lugar errado. O caso que mais
# acontece e o pior de diagnosticar: o segredo existe, esta correto, e o
# toggle DESTE notebook esta desligado.
token = None
motivo_sem_token = None
try:
    from google.colab import userdata
except ImportError:
    motivo_sem_token = 'nao estamos no Colab -- nao ha painel de Secrets.'
else:
    try:
        token = userdata.get('TOKEN_GITHUB')
    except Exception as erro:
        # Comparacao por NOME da classe, e nao por `except userdata.X`, de
        # proposito: essas excecoes nao sao API estavel do Colab, e um
        # AttributeError aqui derrubaria a celula por causa do TRATAMENTO do
        # erro em vez do erro.
        classe = type(erro).__name__
        if classe == 'NotebookAccessError':
            motivo_sem_token = (
                'TOKEN_GITHUB existe, mas ESTE notebook nao tem acesso a ele. '
                'Abra o painel Secrets (icone de chave, barra da esquerda) e '
                'ligue o toggle da linha.'
            )
        elif classe == 'SecretNotFoundError':
            motivo_sem_token = 'nao ha TOKEN_GITHUB no painel Secrets deste Colab.'
        else:
            motivo_sem_token = f'{classe}: {erro}'

if motivo_sem_token:
    print('AVISO -- clone SEM credencial:', motivo_sem_token)

url_publica = f'https://github.com/{DONO}/{NOME_REPO}.git'
url = f'https://{token}@github.com/{DONO}/{NOME_REPO}.git' if token else url_publica

shutil.rmtree(DIR_REPO, ignore_errors=True)
clone = subprocess.run(
    ['git', 'clone', '-q', '--branch', RAMO, url, DIR_REPO],
    capture_output=True, text=True,
)
if clone.returncode != 0:
    # A mensagem do git traz a URL: com token dentro, isso imprimiria o segredo.
    erro = clone.stderr.replace(token, '***') if token else clone.stderr
    raise RuntimeError(
        f'git clone falhou (codigo {clone.returncode}): {erro.strip()}\n'
        'Se for erro de autenticacao, o repositorio e privado: crie um token '
        'em github.com/settings/tokens com escopo `repo`, salve no painel '
        'Secrets do Colab (icone de chave) com o nome TOKEN_GITHUB e ligue o '
        'acesso para este notebook. Alternativa sem token: suba o .zip do '
        'repositorio para /content e descompacte em ' + DIR_REPO + '.'
    )

if token:
    # `git clone` GRAVA a URL com o token em .git/config. O disco do Colab e
    # efemero, mas a saida do notebook e salva no arquivo: um `!git remote -v`
    # numa celula de debug imprimiria o segredo dentro do .ipynb, que e
    # exatamente o vazamento que usar o painel de Secrets existe para evitar.
    subprocess.run(
        ['git', '-C', DIR_REPO, 'remote', 'set-url', 'origin', url_publica],
        check=True,
    )

!pip install -q -e {DIR_REPO}

if DIR_REPO not in sys.path:
    sys.path.insert(0, DIR_REPO)

commit = subprocess.check_output(
    ['git', '-C', DIR_REPO, 'rev-parse', '--short', 'HEAD']
).decode().strip()
print('fraus instalado no commit', commit)

from fraus.avaliacao_ironia import (IRONIAS_SEM_MARCADOR, SINCERAS_COM_MARCADOR,
                                    avaliar_probabilidades)
from fraus.comparacao_modelos import bootstrap_da_diferenca, f1_macro, mcnemar_exato
from fraus.divisao import conferir_divisao, dividir_por_autor, impressao_dos_textos
from fraus.sinais.emocao import NOMES_EMOCOES
from fraus.sinais.ironia import IRONICO, NAO_IRONICO
from fraus.sinais.ironia_laya import PERGUNTA_IRONIA, REVISAO_LAYA_PADRAO
from fraus.treino_laya import (PERGUNTA_EMOCAO, caso_emocao, caso_ironia,
                               distribuicao_de_rotulo, distribuicao_de_votos, mapa_de_poda,
                               probabilidades_emocao, repeticoes_por_classe)

print(len(NOMES_EMOCOES), 'classes:', NOMES_EMOCOES)

## 3. Script oficial de treino e checkpoint base

O treino e o **caminho oficial do Laya**: `finetune_single_device.py`, que
reproduz em uma GPU o laco do notebook 2xT4 do autor -- fine-tuning COMPLETO
(encoder 2,5e-5, cabeca 1e-4, AdamW, cosseno, fp16) com a perda RLCD, que soma
um termo de policy-gradient sobre regras de pontuacao proprias a uma
cross-entropy contra a distribuicao-alvo. Nao ha LoRA nem cabeca congelada.

Duas coisas fixadas, pelo mesmo motivo de sempre (um artefato que muda por
baixo do treino e um treino que ninguem reproduz):

- o **script** vem de um commit imutavel do repositorio e tem o SHA-256
  conferido. Ele so existe a partir da 0.3.23, mas roda contra o pacote
  `laya==0.3.21` instalado -- o mesmo que a API usa -- porque fica numa pasta
  sem o codigo do Laya ao lado;
- o **checkpoint base** e a revisao que `fraus/sinais/ironia_laya.py` declara.

In [ ]:
# 3. Script oficial (commit fixo, hash conferido) e checkpoint base (revisao do Fraus).
import hashlib
import urllib.request

from huggingface_hub import snapshot_download

COMMIT_LAYA = 'd8a2e59781ca135169a36095056132e273cd9938'  # tag v0.3.23
SHA256_SCRIPT = 'c19122f54e3031b885703ccd8b9cf7985f7959bbaefe95bf066b914006eac2e3'
URL_SCRIPT = (f'https://raw.githubusercontent.com/NandhaKishorM/laya/{COMMIT_LAYA}/'
              'research/scripts/finetune_single_device.py')

# A pasta do script NAO pode ter um diretorio `laya/` tres niveis acima: o
# script poe essa raiz no sys.path, e ali ele acharia outra versao do pacote.
DIR_SCRIPT = f'{DIR_TRABALHO}/upstream/research/scripts'
os.makedirs(DIR_SCRIPT, exist_ok=True)
SCRIPT_TREINO = f'{DIR_SCRIPT}/finetune_single_device.py'
with urllib.request.urlopen(URL_SCRIPT) as resposta:
    conteudo_script = resposta.read()
achado = hashlib.sha256(conteudo_script).hexdigest()
assert achado == SHA256_SCRIPT, f'script de treino mudou: sha256 {achado}'
with open(SCRIPT_TREINO, 'wb') as arquivo:
    arquivo.write(conteudo_script)

MODELO_BASE = os.path.join(
    snapshot_download('convaiinnovations/laya', revision=REVISAO_LAYA_PADRAO,
                      allow_patterns=['multilingual/*', 'multilingual/*/*']),
    'multilingual',
)
print('checkpoint base:', MODELO_BASE)

## 4. Corpus de emocao -- o mesmo do notebook 03, com o voto dos anotadores

Mesma fonte (`go_emotions_ptbr`), mesmo mapeamento oficial para Ekman, mesmo
criterio de rotulo (>= 2 anotadores, classe unica) e **a mesma divisao
treino/teste** (15%, semente 42, estratificada). A divisao igual e o que
permite a comparacao pareada: o teste daqui e o teste que o BERTimbau nunca
viu. A celula 8 confere isso em vez de confiar -- ela recalcula a acuracia do
BERTimbau e compara com a que o notebook 03 gravou.

A diferenca esta no ALVO. O BERTimbau treinou com rotulo duro. A perda do Laya
imita uma **distribuicao**, e o GoEmotions publica o voto de cada anotador:
uma fala marcada como raiva por dois e nojo por um vira 0,67 / 0,33 em vez de
"raiva" seca. A discordancia entre anotadores e informacao, e e exatamente o
que alimenta o desprezo derivado. `ALVO_EMOCAO = 'duro'` desliga isso.

O XED-pt entra como no notebook 03: teste independente, sem traducao
automatica, que nenhum dos dois modelos viu.

In [ ]:
# 4. Corpus de emocao, votos por classe de Ekman, divisao do notebook 03 e XED-pt.
import collections
import csv
import io
import json

import pandas as pd
from datasets import Dataset

BASE = 'https://huggingface.co/datasets/antoniomenezes/go_emotions_ptbr/resolve/main'
URL_EKMAN = ('https://raw.githubusercontent.com/google-research/google-research/'
             'master/goemotions/data/ekman_mapping.json')

with urllib.request.urlopen(URL_EKMAN) as resposta:
    mapa_ekman = json.load(resposta)

INGLES_PARA_PT = {
    'joy': 'alegria', 'sadness': 'tristeza', 'anger': 'raiva',
    'fear': 'medo', 'disgust': 'nojo', 'surprise': 'surpresa',
}
fina_para_ekman = {
    fina: INGLES_PARA_PT[grosso]
    for grosso, finas in mapa_ekman.items()
    for fina in finas
}
fina_para_ekman['neutral'] = 'neutro'

bruto = pd.concat(
    [pd.read_csv(f'{BASE}/goemotions_{i}_pt.csv') for i in (1, 2, 3)],
    ignore_index=True,
)
bruto = bruto[~bruto['example_very_unclear'].astype(bool)]
colunas_finas = [c for c in fina_para_ekman if c in bruto.columns]
assert len(colunas_finas) == 28, f'esperava 28 colunas de emocao, achei {len(colunas_finas)}'

LIMIAR_ANOTADORES = 2

votos = bruto.groupby('id')[colunas_finas].sum()
textos = bruto.groupby('id')['texto'].first()

por_ekman = collections.defaultdict(dict)
for coluna in colunas_finas:
    alvo = fina_para_ekman[coluna]
    marcados = votos.index[votos[coluna] >= LIMIAR_ANOTADORES]
    for chave in marcados:
        por_ekman[chave][alvo] = True

# Identico ao notebook 03, mais o `id`: a ordem das linhas decide a divisao.
linhas = [
    {'id': chave, 'texto': textos[chave], 'classe': next(iter(classes))}
    for chave, classes in por_ekman.items()
    if len(classes) == 1
]
dados_emocao = pd.DataFrame(linhas).dropna()
dados_emocao = dados_emocao[dados_emocao['texto'].str.strip().astype(bool)]
dados_emocao['rotulo'] = dados_emocao['classe'].map({n: i for i, n in enumerate(NOMES_EMOCOES)}).astype(int)
dados_emocao = dados_emocao.reset_index(drop=True)

# Votos somados por classe de Ekman, na ordem de NOMES_EMOCOES.
votos_ekman = pd.DataFrame({
    nome: votos[[c for c in colunas_finas if fina_para_ekman[c] == nome]].sum(axis=1)
    for nome in NOMES_EMOCOES
})

conjunto = Dataset.from_pandas(dados_emocao[['texto', 'rotulo']])
conjunto = conjunto.add_column('posicao', list(range(len(conjunto))))
conjunto = conjunto.rename_column('rotulo', 'labels').class_encode_column('labels')
divisao = conjunto.train_test_split(test_size=0.15, seed=42, stratify_by_column='labels')
emocao_treino = dados_emocao.iloc[divisao['train']['posicao']].reset_index(drop=True)
emocao_teste = dados_emocao.iloc[divisao['test']['posicao']].reset_index(drop=True)
assert set(emocao_treino['id']).isdisjoint(set(emocao_teste['id']))
print('emocao: treino', len(emocao_treino), '| teste', len(emocao_teste))
print(emocao_treino['classe'].value_counts())

# XED-pt: Plutchik -> Ekman, descartando anticipation/trust e multirrotulo.
URL_XED = 'https://raw.githubusercontent.com/Helsinki-NLP/XED/master/Projections/pt-projections.tsv'
XED_PARA_PT = {'1': 'raiva', '3': 'nojo', '4': 'medo', '5': 'alegria', '6': 'tristeza', '7': 'surpresa'}
with urllib.request.urlopen(URL_XED) as resposta:
    conteudo = resposta.read().decode('utf-8')
frases_xed, rotulos_xed = [], []
for linha in csv.reader(io.StringIO(conteudo), delimiter='\t'):
    if len(linha) < 2:
        continue
    frase, codigos = linha[0].strip(), [c.strip() for c in linha[1].split(',')]
    classes = {XED_PARA_PT[c] for c in codigos if c in XED_PARA_PT}
    if len(classes) != 1 or not frase:
        continue
    frases_xed.append(frase)
    rotulos_xed.append(NOMES_EMOCOES.index(next(iter(classes))))
print('XED-pt:', len(frases_xed), 'frases')

## 5. Corpus de ironia -- o mesmo do notebook 04

`scripts/preparar_corpus_ironia.py` baixa as fontes publicas do treino do IDPT
em commits imutaveis e tira as hashtags que entregam o rotulo. A divisao segue
a regra do notebook 04: por autor quando ha autor, por fonte quando ha fonte,
estratificada no resto.

**O risco declarado.** O BERTimbau treinado neste corpus aprendeu sentimento
positivo em vez de ironia (ver `docs/treinamento.md`). O Laya treinado no
mesmo dado pode aprender o mesmo atalho. O teste interno nao pega isso -- ele
tem o mesmo vies do treino. Quem pega e a regua de dominio da celula 9, que
nenhum modelo viu e que foi escrita contra exatamente esse atalho.

In [ ]:
# 5. Corpus de ironia preparado e dividido como no notebook 04.
import json
DIR_IRONIA = f'{DIR_TRABALHO}/ironia'
subprocess.run([sys.executable, f'{DIR_REPO}/scripts/preparar_corpus_ironia.py',
                '--destino', DIR_IRONIA], check=True)

dados_ironia = pd.read_csv(f'{DIR_IRONIA}/corpus_ironia_limpo.csv')
dados_ironia = dados_ironia.dropna(subset=['texto', 'rotulo'])
dados_ironia = dados_ironia[dados_ironia['texto'].astype(str).str.strip().astype(bool)]
dados_ironia['rotulo'] = dados_ironia['rotulo'].astype(int)
assert set(dados_ironia['rotulo']) == {NAO_IRONICO, IRONICO}

# A regra mora em fraus/divisao.py e e a MESMA do notebook 04. Linha sem autor
# e grupo proprio: ate 02/10/2026 `astype(str)` fazia do campo vazio o autor
# "nan", e as 18 mil noticias cairam inteiras no teste (63% do corpus, com o
# treino 86% ironico).
FRACAO_TESTE_IRONIA = 0.15
autores = dados_ironia['autor'] if 'autor' in dados_ironia.columns else None
if autores is not None and autores.dropna().astype(str).str.strip().nunique() >= 10:
    treino_idx, teste_idx = dividir_por_autor(autores.tolist(),
                                              fracao_teste=FRACAO_TESTE_IRONIA, semente=42)
    regra_divisao = 'por autor'
elif 'fonte' in dados_ironia.columns and dados_ironia['fonte'].nunique() >= 4:
    from sklearn.model_selection import StratifiedGroupKFold
    treino_idx, teste_idx = next(StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=42)
                                 .split(dados_ironia, dados_ironia['rotulo'],
                                        groups=dados_ironia['fonte']))
    regra_divisao = 'por fonte'
else:
    conjunto_ironia = Dataset.from_pandas(dados_ironia[['texto', 'rotulo']].reset_index(drop=True))
    conjunto_ironia = conjunto_ironia.add_column('posicao', list(range(len(conjunto_ironia))))
    conjunto_ironia = conjunto_ironia.rename_column('rotulo', 'labels').class_encode_column('labels')
    partes = conjunto_ironia.train_test_split(test_size=0.15, seed=42, stratify_by_column='labels')
    treino_idx, teste_idx = partes['train']['posicao'], partes['test']['posicao']
    regra_divisao = 'estratificada'

ironia_treino = dados_ironia.iloc[treino_idx].reset_index(drop=True)
ironia_teste = dados_ironia.iloc[teste_idx].reset_index(drop=True)
print(f'ironia ({regra_divisao}): treino {len(ironia_treino)} | teste {len(ironia_teste)}')
for nome, parte in (('treino', ironia_treino), ('teste', ironia_teste)):
    print(f"{nome}: {(parte['rotulo'] == IRONICO).mean():.1%} ironicas |",
          parte['fonte'].value_counts().to_dict() if 'fonte' in parte.columns else '')
if regra_divisao != 'por fonte':
    conferir_divisao(ironia_treino['rotulo'].tolist(), ironia_teste['rotulo'].tolist(),
                     fracao_teste=FRACAO_TESTE_IRONIA)

# O BERTimbau de ironia tem de ter sido treinado NESTA divisao. Se o teste
# daqui nao e o teste que o notebook 04 gravou, parte dele esteve no treino do
# BERTimbau e a comparacao favorece ele. Conferido antes do treino, que e caro.
impressao_teste_ironia = impressao_dos_textos(ironia_teste['texto'])
caminho_metricas_04 = f'{DIR_MODELOS}/metricas_ironia.json'
gravada = None
if os.path.isfile(caminho_metricas_04):
    with open(caminho_metricas_04, encoding='utf-8') as arquivo:
        gravada = (json.load(arquivo).get('divisao') or {}).get('impressao_teste')
if gravada != impressao_teste_ironia:
    raise RuntimeError(
        'O BERTimbau de ironia do Drive nao foi treinado nesta divisao '
        f'(impressao gravada: {gravada}; daqui: {impressao_teste_ironia}). '
        'Rode o notebook 04 de novo, no mesmo ramo, e volte aqui.'
    )
print('divisao de ironia igual a do notebook 04: impressao', impressao_teste_ironia[:16])

## 6. JSONL de treino

Uma linha por caso, no esquema do Laya (`state` / `questions` / `gold`).

- **Desequilibrio de emocao (43:1).** O script oficial nao aceita peso de
  classe na perda, entao a compensacao vai para o dado: cada classe e repetida
  `raiz(maior / n)` vezes, com teto 4. Raiz, e nao a razao inteira, porque
  repetir `nojo` 43 vezes e ensinar a decorar 444 frases.
- **Ironia** entra com rotulo levemente suavizado (0,05): o corpus foi rotulado
  por hashtag e procedencia, nao por anotador, e alvo 1,0 seco ensina uma
  certeza que o dado nao tem.
- **Texto repetido sai do treino.** Frase que aparece em qualquer conjunto de
  avaliacao (teste interno, XED-pt, regua) e removida do treino do Laya, e a
  celula diz quantas foram.
- **Texto longo e cortado** em `MAX_CARACTERES`. O uso real e mensagem curta de
  atendimento; noticia inteira so encarece o treino.

A regua de dominio (`fraus/avaliacao_ironia.py`) **nao entra aqui**. A celula
confere.

In [ ]:
# 6. JSONL de treino montado com as funcoes testadas de fraus.treino_laya.
import random

ALVO_EMOCAO = 'votos'      # 'votos' = distribuicao dos anotadores | 'duro' = rotulo unico
SUAVIZACAO_IRONIA = 0.05
MAX_CARACTERES = 1200
SEMENTE = 42


def cortar(texto):
    return str(texto)[:MAX_CARACTERES]


repeticoes = repeticoes_por_classe(
    [int((emocao_treino['rotulo'] == i).sum()) for i in range(len(NOMES_EMOCOES))]
)
print('repeticoes por classe:', dict(zip(NOMES_EMOCOES, repeticoes)))

# O GoEmotions repete frases curtas ("obrigado!") com ids diferentes, e a
# divisao do notebook 03 e por id: a mesma frase pode cair nos dois lados.
# Aqui ela sai do TREINO do Laya. O BERTimbau treinou com elas -- a vantagem,
# se houver, e dele, e o XED-pt nao tem esse problema.
textos_de_teste = set(emocao_teste['texto'].map(cortar)) | set(ironia_teste['texto'].map(cortar))
proibidos = (textos_de_teste | set(SINCERAS_COM_MARCADOR) | set(IRONIAS_SEM_MARCADOR)
             | set(frases_xed))

casos, concordam, vistos, removidos = [], 0, 0, 0
for linha in emocao_treino.itertuples():
    if cortar(linha.texto) in proibidos:
        removidos += 1
        continue
    vistos += 1
    if ALVO_EMOCAO == 'votos':
        alvo = distribuicao_de_votos([float(v) for v in votos_ekman.loc[linha.id, NOMES_EMOCOES]])
    else:
        alvo = distribuicao_de_rotulo(linha.rotulo, len(NOMES_EMOCOES))
    concordam += alvo.index(max(alvo)) == linha.rotulo
    casos.extend([caso_emocao(cortar(linha.texto), alvo)] * repeticoes[linha.rotulo])
print(f'alvo de emocao concorda com o rotulo duro em {concordam / vistos:.1%} do treino')
n_emocao = len(casos)

for linha in ironia_treino.itertuples():
    if cortar(linha.texto) in proibidos:
        removidos += 1
        continue
    alvo = distribuicao_de_rotulo(0 if linha.rotulo == IRONICO else 1, 2, suavizacao=SUAVIZACAO_IRONIA)
    casos.append(caso_ironia(cortar(linha.texto), alvo[0]))
print(f'casos de treino: {n_emocao} de emocao + {len(casos) - n_emocao} de ironia')

print(f'{removidos} exemplos removidos do treino por repetirem texto de algum conjunto de avaliacao')

# Nada do que avalia pode estar no treino.
assert not proibidos & {c['state'] for c in casos}, 'frase de avaliacao vazou para o treino'

random.Random(SEMENTE).shuffle(casos)
JSONL_TREINO = f'{DIR_TRABALHO}/treino.jsonl'
with open(JSONL_TREINO, 'w', encoding='utf-8') as arquivo:
    for caso in casos:
        arquivo.write(json.dumps(caso, ensure_ascii=False) + '\n')
print('gravado em', JSONL_TREINO)

## 7. Treino

**Primeiro com `MODO_FUMACA = True`.** Sao 600 casos e uma epoca: minutos. Ele
prova que script, checkpoint e formato se entendem e imprime quanto o treino
completo deve levar nesta GPU. Depois troque para `False` e rode de novo.

**O script oficial nao tem checkpoint por epoca.** Se a sessao do Colab cair
no meio, o treino recomeca -- e o motivo de conferir a estimativa antes. O
resultado so vai para o Drive no fim, copiado inteiro.

A temperatura de calibracao e ajustada pelo proprio script, numa fatia que ele
separa do treino antes de comecar (ate 400 casos). Isso mexe na CONFIANCA, nao
no acerto.

In [ ]:
# 7. Treino com o script oficial do Laya.
import shutil
import time

MODO_FUMACA = True
EPOCAS = 4
CASOS_FUMACA = 600

if MODO_FUMACA:
    dados_treino = f'{DIR_TRABALHO}/fumaca.jsonl'
    with open(dados_treino, 'w', encoding='utf-8') as arquivo:
        for caso in casos[:CASOS_FUMACA]:
            arquivo.write(json.dumps(caso, ensure_ascii=False) + '\n')
    epocas, saida_local = 1, f'{DIR_TRABALHO}/saida-fumaca'
else:
    dados_treino, epocas, saida_local = JSONL_TREINO, EPOCAS, f'{DIR_TRABALHO}/saida'

shutil.rmtree(saida_local, ignore_errors=True)
comando = [sys.executable, SCRIPT_TREINO, '--data', dados_treino, '--model-dir', MODELO_BASE,
           '--output-dir', saida_local, '--device', 'cuda', '--epochs', str(epocas),
           '--seed', str(SEMENTE)]
inicio = time.time()
# Popen com leitura de linha: subprocess.run nao mostra a saida do filho no Colab.
processo = subprocess.Popen(comando, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for linha in processo.stdout:
    print(linha, end='')
if processo.wait() != 0:
    raise RuntimeError(f'treino falhou (codigo {processo.returncode})')
duracao = time.time() - inicio
print(f'\ntreino em {duracao / 60:.1f} min')

if MODO_FUMACA:
    # Inclui o tempo de carregar o modelo, entao superestima um pouco.
    estimativa = duracao * (len(casos) * EPOCAS) / CASOS_FUMACA / 3600
    print(f'estimativa do treino completo ({len(casos)} casos x {EPOCAS} epocas): ate {estimativa:.1f} h')
    print('Se couber na sessao, troque MODO_FUMACA para False e rode esta celula de novo.')
    DIR_LAYA_TREINADO = saida_local
else:
    shutil.copytree(saida_local, DIR_SAIDA, dirs_exist_ok=True)
    DIR_LAYA_TREINADO = DIR_SAIDA
    print('checkpoint copiado para', DIR_SAIDA)

## 8. Comparacao de emocao

Tres modelos, os mesmos exemplos:

- **BERTimbau** do notebook 03 (lido do Drive);
- **Laya sem treino** -- o que o Fraus tem hoje, respondendo a pergunta a frio;
- **Laya treinado** nesta sessao.

Dois conjuntos: o teste interno (mesma traducao automatica do treino) e o
**XED-pt**, que e a medida honesta. O numero de referencia do BERTimbau no
cartao do modelo e F1-macro 0,584 no interno e **0,281 no XED-pt**.

Para cada conjunto: F1-macro e acuracia de cada um, McNemar exato do Laya
treinado contra o BERTimbau e IC 95% da diferenca de F1-macro. **Diferenca cujo
intervalo cruza o zero nao e vitoria de ninguem.**

A celula tambem confere se a divisao reproduziu a do notebook 03: se a
acuracia do BERTimbau aqui nao bater com a que ele gravou, o teste interno
pode conter exemplos que ele viu no treino, e so o XED-pt vale.

In [ ]:
# 8. Emocao: BERTimbau x Laya sem treino x Laya treinado.
import laya
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoModelForSequenceClassification, AutoTokenizer

for pasta in (DIR_BERT_EMOCAO, DIR_BERT_IRONIA):
    if not os.path.isfile(f'{pasta}/config.json'):
        raise FileNotFoundError(
            f'BERTimbau nao encontrado em {pasta}. A comparacao precisa dos artefatos dos '
            'notebooks 03 e 04 no Drive -- sem eles nao ha contra quem comparar.'
        )


def prever_bertimbau(pasta, textos, lote=64):
    # Mesmo corte de 192 tokens de fraus/sinais/texto.py, mas em GPU.
    tokenizador = AutoTokenizer.from_pretrained(pasta)
    modelo = AutoModelForSequenceClassification.from_pretrained(pasta).to('cuda').eval()
    saida = []
    with torch.inference_mode():
        for i in range(0, len(textos), lote):
            entradas = tokenizador(textos[i:i + lote], truncation=True, max_length=192,
                                   padding=True, return_tensors='pt').to('cuda')
            saida.extend(torch.softmax(modelo(**entradas).logits, dim=-1).tolist())
    del modelo
    torch.cuda.empty_cache()
    return saida


def prever_laya(agente, textos, pergunta, leitor, lote=64):
    saida = []
    for i in range(0, len(textos), lote):
        respostas = agente.predict_batch(textos[i:i + lote], pergunta, lang='pt')
        saida.extend(leitor(resposta) for resposta in respostas)
    return saida


def ler_ironia(resposta):
    p = float(resposta['answers']['ironia']['probabilities']['A'])
    return [1.0 - p, p]


def argmax(probabilidades):
    return [int(np.argmax(p)) for p in probabilidades]


def comparar(nome, rotulos, preditos_por_modelo, classes, nomes_classes):
    print(f'\n===== {nome} ({len(rotulos)} exemplos) =====')
    resumo = {}
    for modelo, preditos in preditos_por_modelo.items():
        resumo[modelo] = {
            'f1_macro': f1_macro(rotulos, preditos, classes=classes),
            'acuracia': float(accuracy_score(rotulos, preditos)),
        }
        print(f"{modelo:16s} F1-macro {resumo[modelo]['f1_macro']:.4f} | acuracia {resumo[modelo]['acuracia']:.4f}")
    a, b = preditos_por_modelo['laya_treinado'], preditos_por_modelo['bertimbau']
    mc = mcnemar_exato([p == r for p, r in zip(a, rotulos)], [p == r for p, r in zip(b, rotulos)])
    bs = bootstrap_da_diferenca(rotulos, a, b, classes=classes)
    print(f'Laya treinado - BERTimbau: dF1-macro {bs.diferenca:+.4f} '
          f'(IC 95% {bs.ic_inferior:+.4f} a {bs.ic_superior:+.4f})')
    print(f'McNemar exato: so o Laya acerta {mc.so_a}, so o BERTimbau acerta {mc.so_b}, p = {mc.p_valor:.4g}')
    if bs.ic_inferior <= 0 <= bs.ic_superior:
        print('-> o intervalo cruza o zero: NAO ha diferenca demonstrada neste conjunto.')
    print(classification_report(rotulos, a, labels=classes, target_names=nomes_classes,
                                digits=4, zero_division=0))
    resumo['diferenca_f1_macro'] = {'valor': bs.diferenca, 'ic95': [bs.ic_inferior, bs.ic_superior]}
    resumo['mcnemar'] = {'so_laya': mc.so_a, 'so_bertimbau': mc.so_b, 'p_valor': mc.p_valor}
    return resumo


agente_treinado = laya.load(DIR_LAYA_TREINADO, device='cuda')
agente_base = laya.load(MODELO_BASE, device='cuda')

textos_teste = [cortar(t) for t in emocao_teste['texto']]
rotulos_teste = emocao_teste['rotulo'].tolist()
classes_emocao = list(range(len(NOMES_EMOCOES)))

pred_emocao = {
    'bertimbau': argmax(prever_bertimbau(DIR_BERT_EMOCAO, textos_teste)),
    'laya_sem_treino': argmax(prever_laya(agente_base, textos_teste, PERGUNTA_EMOCAO, probabilidades_emocao)),
    'laya_treinado': argmax(prever_laya(agente_treinado, textos_teste, PERGUNTA_EMOCAO, probabilidades_emocao)),
}
metricas = {'emocao_interno': comparar('emocao -- teste interno', rotulos_teste, pred_emocao,
                                       classes_emocao, NOMES_EMOCOES)}

# A divisao reproduziu a do notebook 03?
caminho_metricas_03 = f'{DIR_MODELOS}/metricas_emocao.json'
if os.path.isfile(caminho_metricas_03):
    gravada = json.load(open(caminho_metricas_03))['acuracia']
    agora = metricas['emocao_interno']['bertimbau']['acuracia']
    print(f'BERTimbau: acuracia gravada pelo notebook 03 = {gravada:.4f}, recalculada aqui = {agora:.4f}')
    metricas['emocao_interno']['divisao_reproduzida'] = abs(gravada - agora) <= 0.005
    if not metricas['emocao_interno']['divisao_reproduzida']:
        print('AVISO: a divisao NAO bateu com a do notebook 03. O teste interno pode conter '
              'exemplos que o BERTimbau viu no treino; use so o XED-pt para comparar.')
else:
    print('AVISO: metricas_emocao.json ausente -- nao da para conferir se a divisao e a mesma.')
    metricas['emocao_interno']['divisao_reproduzida'] = None

presentes = sorted(set(rotulos_xed))
pred_xed = {
    'bertimbau': argmax(prever_bertimbau(DIR_BERT_EMOCAO, frases_xed)),
    'laya_sem_treino': argmax(prever_laya(agente_base, frases_xed, PERGUNTA_EMOCAO, probabilidades_emocao)),
    'laya_treinado': argmax(prever_laya(agente_treinado, frases_xed, PERGUNTA_EMOCAO, probabilidades_emocao)),
}
metricas['emocao_xed_pt'] = comparar('emocao -- XED-pt (independente)', rotulos_xed, pred_xed,
                                     presentes, [NOMES_EMOCOES[i] for i in presentes])

## 9. Comparacao de ironia

O teste interno compara os tres modelos no dado do IDPT. A **regua de dominio**
e o que decide: dez falas sinceras com marcador conversacional e dez ironias
sem marcador, escritas contra os dois atalhos conhecidos. Ela e pequena e
autoral -- os numeros dela sao *taxas na regua*, nunca acuracia externa -- mas
e o unico conjunto aqui com cara de atendimento.

Um modelo que marca 95% no teste interno e erra metade da regua aprendeu o
corpus, nao a ironia.

In [ ]:
# 9. Ironia: teste interno e regua de dominio.
textos_ironia = [cortar(t) for t in ironia_teste['texto']]
rotulos_ironia = ironia_teste['rotulo'].tolist()

prob_ironia = {
    'bertimbau': prever_bertimbau(DIR_BERT_IRONIA, textos_ironia),
    'laya_sem_treino': prever_laya(agente_base, textos_ironia, PERGUNTA_IRONIA, ler_ironia),
    'laya_treinado': prever_laya(agente_treinado, textos_ironia, PERGUNTA_IRONIA, ler_ironia),
}
metricas['ironia_interno'] = comparar(
    'ironia -- teste interno', rotulos_ironia,
    {modelo: argmax(p) for modelo, p in prob_ironia.items()},
    [NAO_IRONICO, IRONICO], ['nao-ironico', 'ironico'],
)

# Nas noticias a fonte ENTREGA o rotulo (Estadao e sempre nao-ironico; os sites
# de satira, sempre ironicos): acertar ali pode ser reconhecer o jornal. Os
# tweets sao o recorte em que os dois rotulos vem do mesmo meio.
recortes_ironia = {}
if 'fonte' in ironia_teste.columns:
    fontes_teste = ironia_teste['fonte'].astype(str).tolist()
    for meio in ('tweets', 'noticias'):
        posicoes = [i for i, f in enumerate(fontes_teste) if f.startswith(meio)]
        if not posicoes or len({rotulos_ironia[i] for i in posicoes}) < 2:
            continue
        recortes_ironia[meio] = {
            'rotulos': [rotulos_ironia[i] for i in posicoes],
            'preditos': {modelo: [int(np.argmax(p[i])) for i in posicoes]
                         for modelo, p in prob_ironia.items()},
        }
        metricas[f'ironia_{meio}'] = comparar(
            f'ironia -- so {meio}', recortes_ironia[meio]['rotulos'],
            recortes_ironia[meio]['preditos'], [NAO_IRONICO, IRONICO], ['nao-ironico', 'ironico'])

print('\n===== regua de dominio (10 sinceras + 10 ironicas, nenhuma no treino) =====')
metricas['ironia_regua'] = {}
sinceras, ironicas = list(SINCERAS_COM_MARCADOR), list(IRONIAS_SEM_MARCADOR)
leitores = {
    'bertimbau': lambda textos: prever_bertimbau(DIR_BERT_IRONIA, textos),
    'laya_sem_treino': lambda textos: prever_laya(agente_base, textos, PERGUNTA_IRONIA, ler_ironia),
    'laya_treinado': lambda textos: prever_laya(agente_treinado, textos, PERGUNTA_IRONIA, ler_ironia),
}
for modelo, ler in leitores.items():
    resultado = avaliar_probabilidades([p[IRONICO] for p in ler(sinceras)],
                                       [p[IRONICO] for p in ler(ironicas)])
    metricas['ironia_regua'][modelo] = resultado.como_dict()
    print(f'{modelo:16s} falso positivo {resultado.taxa_falso_positivo:.0%} | '
          f'falso negativo {resultado.taxa_falso_negativo:.0%} | '
          f'maior confianca errada {resultado.maior_confianca_errada:.2f}')

## 10. Encolhimento para o deploy

A API roda na Vercel, onde o pacote ja esta perto do teto. O checkpoint
multilingue tem **322 M de parametros, e 197 M deles (61%) sao a tabela de
embeddings de 256 mil tokens** -- chines, arabe, cirilico. Um modelo que so le
portugues usa uma fracao disso. Tres passos, do mais seguro para o mais
arriscado:

1. **Poda de vocabulario** -- fica so a linha de embedding dos tokens que
   aparecem em texto portugues (os corpora deste notebook, as duas perguntas,
   os lexicos do Fraus). E SEM PERDA para texto coberto: a conta e a mesma, so
   a tabela e menor. Token fora da lista vira `<unk>`; a celula mede quanto do
   XED-pt (texto que ninguem usou para montar a lista) cai nesse caso. O mapa
   id-original -> linha vai DENTRO do grafo ONNX, entao tokenizador e
   `ONNXAgent` nao mudam.
2. **Pesos em FP16 no disco** -- o checkpoint do Laya ja e gravado em meia
   precisao; o export para ONNX so o reescreve em float32. Voltar a float16 no
   arquivo, com a conversao para float32 na carga, corta o tamanho pela metade
   sem mudar a conta. So entra com ZERO decisao trocada.
3. **INT8 dinamico** nos MatMul -- o mesmo do notebook 06. Corta o resto por
   quatro, mas MEXE nas probabilidades. So substitui o FP32 se trocar no
   maximo 1% das decisoes numa amostra do teste; senao o artefato sai FP32
   podado e o manifesto diz isso.

O grafo exportado so aceita **uma frase e uma pergunta por chamada**
(limitacao do export do upstream, ja tratada em `ClassificadorIroniaLayaOnnx`).
Emocao + ironia sao duas passadas por mensagem.

In [ ]:
# 10. Poda de vocabulario, export ONNX, pesos em FP16, INT8 com portao e medicao.
!pip -q install 'protobuf>=5.29.1,<6' 'onnx>=1.17' 'onnxruntime>=1.20' 'onnxscript>=0.1'

import glob
from pathlib import Path

import torch.nn as nn
from laya.agent import Agent

AMOSTRA_PORTAO = 150      # frases do teste por tarefa, uma chamada ONNX por frase
MAX_TROCAS_INT8 = 0.01

agente_cpu = Agent(DIR_LAYA_TREINADO, compile=False, device='cpu')
tokenizador = agente_cpu.tok
perguntas = {**PERGUNTA_EMOCAO, **PERGUNTA_IRONIA}

# --- quais tokens ficam -------------------------------------------------
textos_vocabulario = (dados_emocao['texto'].astype(str).tolist()
                      + dados_ironia['texto'].astype(str).map(cortar).tolist()
                      + list(SINCERAS_COM_MARCADOR) + list(IRONIAS_SEM_MARCADOR))
for caminho in glob.glob(f'{DIR_REPO}/fraus/dados/*.csv'):
    textos_vocabulario.append(open(caminho, encoding='utf-8', errors='ignore').read())

ids_mantidos = set(tokenizador.all_special_ids)
for i in range(0, len(textos_vocabulario), 512):
    lote = [t[:20000] for t in textos_vocabulario[i:i + 512]]
    for ids in tokenizador(lote, add_special_tokens=True)['input_ids']:
        ids_mantidos.update(ids)
# O cabecalho que o Laya monta: "choice question: <instrucao>" e " <chave>: <criterio>".
for pergunta in perguntas.values():
    ids_mantidos.update(tokenizador('choice question: ' + pergunta['instructions'])['input_ids'])
    for chave, criterio in pergunta['criteria'].items():
        ids_mantidos.update(tokenizador(f' {chave}: {criterio}', add_special_tokens=False)['input_ids'])

mapa, mantidos = mapa_de_poda(ids_mantidos, tamanho=len(tokenizador),
                              id_desconhecido=tokenizador.unk_token_id)
print(f'vocabulario: {len(mantidos)} de {len(tokenizador)} tokens ficam')

# Cobertura em texto que NAO entrou na lista: quanto do XED-pt vira <unk>?
fora = total = 0
for ids in tokenizador(frases_xed, add_special_tokens=False)['input_ids']:
    total += len(ids)
    fora += sum(1 for i in ids if i not in ids_mantidos)
cobertura_xed = 1 - fora / total
print(f'XED-pt: {cobertura_xed:.2%} dos tokens cobertos ({fora} de {total} viram <unk>)')


class EmbeddingPodado(nn.Module):
    """Tabela so com as linhas mantidas; o mapa traduz o id original."""

    def __init__(self, original, mantidos, mapa):
        super().__init__()
        self.tabela = nn.Embedding.from_pretrained(
            original.weight.detach()[torch.tensor(mantidos)].clone(), freeze=True)
        self.register_buffer('mapa', torch.tensor(mapa, dtype=torch.long))

    def forward(self, input_ids):
        return self.tabela(self.mapa[input_ids])


# --- referencia em PyTorch, ANTES da poda --------------------------------
def amostra(textos, rotulos, n):
    posicoes = random.Random(SEMENTE).sample(range(len(textos)), min(n, len(textos)))
    return [textos[i] for i in posicoes], [rotulos[i] for i in posicoes]


frases_emocao, gabarito_emocao = amostra(textos_teste, rotulos_teste, AMOSTRA_PORTAO)
frases_ironia, gabarito_ironia = amostra(textos_ironia, rotulos_ironia, AMOSTRA_PORTAO)


def decisoes(agente):
    # Uma frase e UMA pergunta por chamada: e o que o grafo exportado aceita.
    emocao = [int(np.argmax(probabilidades_emocao(
        agente.predict_batch([f], PERGUNTA_EMOCAO, lang='pt')[0]))) for f in frases_emocao]
    ironia = [int(np.argmax(ler_ironia(
        agente.predict_batch([f], PERGUNTA_IRONIA, lang='pt')[0]))) for f in frases_ironia]
    return emocao, ironia


ref_emocao, ref_ironia = decisoes(agente_cpu)

embeddings = agente_cpu.model.encoder.embeddings
parametros_antes = sum(p.numel() for p in agente_cpu.model.parameters())
embeddings.tok_embeddings = EmbeddingPodado(embeddings.tok_embeddings, mantidos, mapa)
agente_cpu.model.float().eval()
parametros_depois = sum(p.numel() for p in agente_cpu.model.parameters())
print(f'parametros: {parametros_antes / 1e6:.1f} M -> {parametros_depois / 1e6:.1f} M')

# --- export -----------------------------------------------------------------
RAIZ_ZIP = Path(f'{DIR_TRABALHO}/pacote')
DESTINO = RAIZ_ZIP / 'modelos-onnx' / 'laya-ironia'   # a pasta que a API ja le
shutil.rmtree(RAIZ_ZIP, ignore_errors=True)
DESTINO.mkdir(parents=True)
entradas = (
    torch.randint(5, 100, (1, 16), dtype=torch.long),
    torch.ones((1, 16), dtype=torch.long),
    torch.tensor([[1, 5]], dtype=torch.long),
    torch.tensor([[True, True]], dtype=torch.bool),
    torch.tensor([0], dtype=torch.long),
)
fp32 = DESTINO / 'laya.onnx'
torch.onnx.export(
    agente_cpu.model, entradas, str(fp32), export_params=True, opset_version=18,
    do_constant_folding=True,
    input_names=['input_ids', 'attention_mask', 'marker_pos', 'marker_mask', 'qtype'],
    output_names=['logits', 'act_logits'],
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'seq_len'}, 'attention_mask': {0: 'batch_size', 1: 'seq_len'},
        'marker_pos': {0: 'batch_size', 1: 'num_markers'}, 'marker_mask': {0: 'batch_size', 1: 'num_markers'},
        'qtype': {0: 'batch_size'}, 'logits': {0: 'batch_size', 1: 'num_markers'},
        'act_logits': {0: 'batch_size'}},
)
shutil.copy2(f'{DIR_LAYA_TREINADO}/rl_agent_config.json', DESTINO / 'rl_agent_config.json')
shutil.copytree(f'{DIR_LAYA_TREINADO}/tokenizer', DESTINO / 'tokenizer')
shutil.copytree(f'{DIR_LAYA_TREINADO}/encoder', DESTINO / 'encoder')

import onnx
from laya.onnx_agent import ONNXAgent
from onnxruntime.quantization import QuantType, quantize_dynamic

grafo = onnx.load(str(fp32))
del grafo.graph.value_info[:]
int8 = DESTINO / 'laya.int8.candidato.onnx'
quantize_dynamic(model_input=grafo, model_output=str(int8), op_types_to_quantize=['MatMul'],
                 weight_type=QuantType.QInt8, per_channel=True)


def tamanho_mb(*arquivos):
    return sum(Path(a).stat().st_size for a in arquivos if Path(a).exists()) / 1024**2


# Pesos em float16 NO DISCO, convertidos para float32 na carga: nao e
# quantizacao, a conta roda nos mesmos numeros (ver fraus/pesos_fp16.py).
from fraus.pesos_fp16 import guardar_em_fp16

fp16 = DESTINO / 'laya.fp16.candidato.onnx'
tensores_fp16, perda_fp16 = guardar_em_fp16(fp32, fp16)
print(f'FP16: {tensores_fp16} tensores, maior diferenca de peso {perda_fp16:.2e}')

mb_fp32 = tamanho_mb(fp32, str(fp32) + '.data')
mb_fp16 = tamanho_mb(fp16)
mb_int8 = tamanho_mb(int8)
print(f'FP32 podado: {mb_fp32:.0f} MB | FP16 em disco: {mb_fp16:.0f} MB | INT8: {mb_int8:.0f} MB')


def conferir(caminho, nome):
    emocao, ironia = decisoes(ONNXAgent(str(DESTINO), onnx_path=str(caminho)))
    trocas = (sum(a != b for a, b in zip(ref_emocao, emocao))
              + sum(a != b for a, b in zip(ref_ironia, ironia)))
    taxa = trocas / (len(emocao) + len(ironia))
    acerto_emocao = float(np.mean([a == b for a, b in zip(emocao, gabarito_emocao)]))
    acerto_ironia = float(np.mean([a == b for a, b in zip(ironia, gabarito_ironia)]))
    print(f'{nome}: {trocas} decisoes trocadas contra o PyTorch ({taxa:.2%}) | '
          f'acuracia na amostra: emocao {acerto_emocao:.3f}, ironia {acerto_ironia:.3f}')
    return taxa


trocas_fp32 = conferir(fp32, 'ONNX FP32 podado')
trocas_fp16 = conferir(fp16, 'ONNX FP16 em disco, podado')
trocas_int8 = conferir(int8, 'ONNX INT8 podado')

# A poda nao pode mudar decisao em texto coberto; se mudou, e <unk> no teste
# (cobertura baixa) ou erro de export -- nos dois casos o artefato nao sai.
assert trocas_fp32 <= MAX_TROCAS_INT8, (
    f'a poda/export trocou {trocas_fp32:.2%} das decisoes; amplie textos_vocabulario')

# Do menor para o maior, fica o primeiro que nao muda decisao. O FP16 so
# entra com ZERO troca: ele nao tem por que mudar nada, e se mudou ha erro.
if trocas_int8 <= MAX_TROCAS_INT8:
    formato, escolhido = 'int8', int8
elif trocas_fp16 == 0:
    formato, escolhido = 'fp16', fp16
    print(f'INT8 REPROVADO ({trocas_int8:.2%} > {MAX_TROCAS_INT8:.0%}): o artefato sai FP16 podado.')
else:
    formato, escolhido = 'fp32', None
    print(f'INT8 ({trocas_int8:.2%}) e FP16 ({trocas_fp16:.2%}) reprovados: o artefato sai FP32 podado.')
for candidato in (int8, fp16):
    if candidato != escolhido:
        candidato.unlink()
if escolhido is not None:
    fp32.unlink()
    Path(str(fp32) + '.data').unlink(missing_ok=True)
    escolhido.rename(fp32)
print('formato do artefato:', formato, f'| {tamanho_mb(fp32, str(fp32) + ".data"):.0f} MB')

## 11. Metricas e pacote

O `metricas_laya.json` vai para a raiz de `modelos/`, irmao dos outros. Ele
carrega a procedencia junto: revisao do checkpoint base, commit do script,
commit do `fraus`, alvo usado e se foi rodada de fumaca -- **numero de rodada
de fumaca nao e resultado** e o arquivo diz isso.

O ZIP tem o mesmo desenho do notebook 06 (`modelos-onnx/laya-ironia/` com
`manifesto.json`), entao a cabeca de ironia da API o le sem mudanca. A leitura
de EMOCAO pelo Laya em producao ainda nao existe no `fraus`: promover este
checkpoint para a emocao e uma decisao a tomar com estes numeros na mao, nao
um efeito colateral do notebook.

In [ ]:
# 11. Metricas, manifesto e ZIP.
import zipfile

metricas['procedencia'] = {
    'rodada_de_fumaca': MODO_FUMACA,
    'checkpoint_base': f'convaiinnovations/laya@{REVISAO_LAYA_PADRAO} (multilingual)',
    'script_treino': f'NandhaKishorM/laya@{COMMIT_LAYA} finetune_single_device.py',
    'laya': '0.3.21',
    'fraus_commit': commit,
    'epocas': 1 if MODO_FUMACA else EPOCAS,
    'alvo_emocao': ALVO_EMOCAO,
    'suavizacao_ironia': SUAVIZACAO_IRONIA,
    'casos_treino': CASOS_FUMACA if MODO_FUMACA else len(casos),
    'corpus_emocao': 'go_emotions_ptbr (traducao automatica), Ekman, >= 2 anotadores',
    'corpus_ironia': f'fontes publicas do treino IDPT, divisao {regra_divisao}',
    'impressao_teste_ironia': impressao_teste_ironia,
}
metricas['encolhimento'] = {
    'tokens_mantidos': len(mantidos),
    'tokens_originais': len(tokenizador),
    'cobertura_xed_pt': cobertura_xed,
    'parametros_m': parametros_depois / 1e6,
    'formato': formato,
    'mb': tamanho_mb(fp32, str(fp32) + '.data'),
    'trocas_fp32': trocas_fp32,
    'trocas_fp16': trocas_fp16,
    'trocas_int8': trocas_int8,
}
destino_metricas = (f'{DIR_TRABALHO}/metricas_laya_fumaca.json' if MODO_FUMACA
                    else f'{DIR_MODELOS}/metricas_laya.json')
with open(destino_metricas, 'w', encoding='utf-8') as arquivo:
    json.dump(metricas, arquivo, ensure_ascii=False, indent=2)
print('metricas em', destino_metricas)

manifesto = {
    'schema': 1,
    'modelo': 'convaiinnovations/laya (multilingual) + fine-tune Fraus',
    'revisao': REVISAO_LAYA_PADRAO,
    'arquivo': 'laya.onnx',
    'formato': formato,
    'sha256': hashlib.sha256(fp32.read_bytes()).hexdigest(),
    'perguntas': sorted(perguntas),
    'tokens_mantidos': len(mantidos),
    'trocas_int8': trocas_int8,
    'rodada_de_fumaca': MODO_FUMACA,
}
(DESTINO / 'manifesto.json').write_text(json.dumps(manifesto, ensure_ascii=False, indent=2) + '\n')

if MODO_FUMACA:
    print('Rodada de fumaca: o caminho inteiro funcionou. NAO ha ZIP -- treine de verdade antes.')
else:
    zip_saida = f'{DIR_MODELOS}/laya-fraus-onnx.zip'
    with zipfile.ZipFile(zip_saida, 'w', zipfile.ZIP_DEFLATED) as pacote:
        for caminho in DESTINO.rglob('*'):
            if caminho.is_file():
                pacote.write(caminho, caminho.relative_to(RAIZ_ZIP))
    print('pacote em', zip_saida, f'({os.path.getsize(zip_saida) / 1024**2:.0f} MB)')
    print('sha256:', hashlib.sha256(open(zip_saida, 'rb').read()).hexdigest())

## 12. Laudo de comparacao para a dashboard

Esta celula junta o que as celulas 8, 9 e 10 ja calcularam e grava
`comparacao_modelos.json` -- o arquivo que a aba **Modelo > Comparacao** da
dashboard le -- e o mesmo conteudo por extenso em `comparacao-modelos.md`.
Nada e treinado nem previsto de novo, exceto a regua (20 frases) e a medicao
de tempo.

**Matriz de confusao e VP/FP/FN/VN** saem por classe, uma contra o resto. Em
ironia a classe positiva e `ironico`.

**Tempo.** A GPU do Colab nao e a producao, entao a latencia e medida em
**CPU**, uma mensagem por vez, numa amostra fixa. O BERTimbau roda em PyTorch e
o Laya no artefato ONNX da celula 10 -- sao os executores disponiveis aqui, e
o laudo registra qual foi. O numero vale para esta maquina; ele serve para
comparar um modelo com o outro, nao para prever o tempo da Vercel.

Copie `comparacao_modelos.json` para `modelos/` na maquina que roda a API.

In [ ]:
# 12. Laudo de comparacao (dashboard) e relatorio em Markdown.
import importlib
import inspect
import platform
import statistics
import traceback
import urllib.request
from datetime import datetime, timezone

# As funcoes do laudo vem de um commit FIXO do repositorio: a sessao pode ter
# sido aberta antes de elas existirem, e URL por commit nao sofre cache.
COMMIT_LAUDO = 'b403c5d5ae4fcaad3b1f86262cc689b734c4509e'
import fraus.comparacao_modelos as _comparacao
if 'classes_do_f1' not in inspect.signature(getattr(_comparacao, 'avaliar_conjunto', lambda: 0)).parameters:
    _url = (f'https://raw.githubusercontent.com/Joao-Tavares0312/Fraus/{COMMIT_LAUDO}/'
            'fraus/comparacao_modelos.py')
    with urllib.request.urlopen(_url) as _resposta:
        _fonte = _resposta.read()
    with open(_comparacao.__file__, 'wb') as _arquivo:
        _arquivo.write(_fonte)
    _comparacao = importlib.reload(_comparacao)
avaliar_conjunto = _comparacao.avaliar_conjunto
montar_laudo = _comparacao.montar_laudo
relatorio_markdown = _comparacao.relatorio_markdown

AMOSTRA_LATENCIA = 50
AQUECIMENTO = 3

# --- conjuntos ---------------------------------------------------------------
rotulos_regua = [NAO_IRONICO] * len(sinceras) + [IRONICO] * len(ironicas)
pred_regua = {
    modelo: [IRONICO if p[IRONICO] > 0.5 else NAO_IRONICO for p in ler(sinceras + ironicas)]
    for modelo, ler in leitores.items()
}
conjuntos = [
    avaliar_conjunto(
        identificador='emocao_interno', tarefa='emocao', nome='Emoção — teste interno',
        independente=False, rotulos=rotulos_teste, preditos_por_modelo=pred_emocao,
        classes=classes_emocao, nomes_classes=NOMES_EMOCOES),
    avaliar_conjunto(
        identificador='emocao_xed_pt', tarefa='emocao', nome='Emoção — XED-pt',
        independente=True, rotulos=rotulos_xed, preditos_por_modelo=pred_xed,
        # O XED-pt nao tem `neutro`, mas o modelo pode responder `neutro`: a
        # matriz tem as sete classes e o F1 conta so as que o conjunto contem.
        classes=classes_emocao, nomes_classes=NOMES_EMOCOES, classes_do_f1=presentes),
    avaliar_conjunto(
        identificador='ironia_interno', tarefa='ironia', nome='Ironia — teste interno',
        independente=False, rotulos=rotulos_ironia,
        preditos_por_modelo={m: argmax(p) for m, p in prob_ironia.items()},
        classes=[NAO_IRONICO, IRONICO], nomes_classes=['nao-ironico', 'ironico']),
    *[
        avaliar_conjunto(
            identificador=f'ironia_{meio}', tarefa='ironia',
            nome={'tweets': 'Ironia — só tweets', 'noticias': 'Ironia — só notícias'}[meio],
            independente=False, rotulos=recorte['rotulos'],
            preditos_por_modelo=recorte['preditos'],
            classes=[NAO_IRONICO, IRONICO], nomes_classes=['nao-ironico', 'ironico'])
        for meio, recorte in globals().get('recortes_ironia', {}).items()
    ],
    avaliar_conjunto(
        identificador='ironia_regua', tarefa='ironia', nome='Ironia — régua de domínio',
        independente=True, rotulos=rotulos_regua, preditos_por_modelo=pred_regua,
        classes=[NAO_IRONICO, IRONICO], nomes_classes=['nao-ironico', 'ironico']),
]
for conjunto in conjuntos:
    c = conjunto['comparacao']
    print(f"{conjunto['nome']}: dF1 {c['diferenca_f1_macro']:+.4f} "
          f"(IC {c['ic95'][0]:+.4f} a {c['ic95'][1]:+.4f}) "
          f"{'diferenca demonstrada' if c['diferenca_demonstrada'] else 'SEM diferenca demonstrada'}")


# --- tempo em CPU, uma mensagem por vez ----------------------------------------
def cronometrar(chamar, frases):
    for frase in frases[:AQUECIMENTO]:
        chamar(frase)
    tempos = []
    for frase in frases:
        inicio = time.perf_counter()
        chamar(frase)
        tempos.append((time.perf_counter() - inicio) * 1000)
    tempos.sort()
    return {'mediana_ms': statistics.median(tempos),
            'p95_ms': tempos[min(len(tempos) - 1, int(0.95 * len(tempos)))]}


def bertimbau_em_cpu(pasta):
    tok = AutoTokenizer.from_pretrained(pasta)
    modelo_cpu = AutoModelForSequenceClassification.from_pretrained(pasta).eval()

    def chamar(frase):
        with torch.inference_mode():
            modelo_cpu(**tok([frase], truncation=True, max_length=192, return_tensors='pt'))
    return chamar


# A medicao de tempo NAO pode custar o laudo: se ela falhar, o laudo sai com
# `latencia: null` -- "nao foi medida", nunca zero -- e o erro fica impresso.
latencia = None
try:
    frases_tempo_emocao, _ = amostra(textos_teste, rotulos_teste, AMOSTRA_LATENCIA)
    frases_tempo_ironia, _ = amostra(textos_ironia, rotulos_ironia, AMOSTRA_LATENCIA)
    laya_onnx = ONNXAgent(str(DESTINO), onnx_path=str(fp32))
    medidas = []
    for tarefa, pasta, pergunta, frases in (
        ('emocao', DIR_BERT_EMOCAO, PERGUNTA_EMOCAO, frases_tempo_emocao),
        ('ironia', DIR_BERT_IRONIA, PERGUNTA_IRONIA, frases_tempo_ironia),
    ):
        medidas.append({'modelo': 'bertimbau', 'tarefa': tarefa, 'executor': 'torch',
                        **cronometrar(bertimbau_em_cpu(pasta), frases)})
        medidas.append({'modelo': 'laya_treinado', 'tarefa': tarefa, 'executor': f'onnx-{formato}',
                        **cronometrar(lambda f, q=pergunta: laya_onnx.predict_batch([f], q, lang='pt'),
                                      frases)})
    latencia = {
        'hardware': f'CPU do Colab ({os.cpu_count()} núcleos, {platform.machine()})',
        'amostra': AMOSTRA_LATENCIA,
        'medidas': medidas,
    }
    for medida in medidas:
        print(f"{medida['modelo']:14s} {medida['tarefa']:7s} {medida['executor']:10s} "
              f"mediana {medida['mediana_ms']:.0f} ms | p95 {medida['p95_ms']:.0f} ms")
except Exception:
    print('AVISO: a latencia NAO foi medida; o laudo sai sem ela. Erro:')
    traceback.print_exc()

# --- laudo ---------------------------------------------------------------------
laudo = montar_laudo(
    conjuntos=conjuntos, latencia=latencia,
    procedencia=(metricas.get('procedencia') if isinstance(globals().get('metricas'), dict) else None)
    or {'observacao': 'celula 11 nao executada: procedencia nao registrada'},
    rodada_de_fumaca=MODO_FUMACA,
    gerado_em=datetime.now(timezone.utc).isoformat(timespec='seconds'),
)
pasta_laudo = DIR_TRABALHO if MODO_FUMACA else DIR_MODELOS
caminho_laudo = f'{pasta_laudo}/comparacao_modelos.json'
with open(caminho_laudo, 'w', encoding='utf-8') as arquivo:
    json.dump(laudo, arquivo, ensure_ascii=False, indent=2)
caminho_relatorio = f'{pasta_laudo}/comparacao-modelos.md'
with open(caminho_relatorio, 'w', encoding='utf-8') as arquivo:
    arquivo.write(relatorio_markdown(laudo))
print('laudo em', caminho_laudo)
print('relatorio em', caminho_relatorio)
if MODO_FUMACA:
    print('Rodada de fumaca: o laudo sai marcado como fumaca e a dashboard avisa.')
try:
    from google.colab import files
    files.download(caminho_laudo)
    files.download(caminho_relatorio)
except Exception as erro:
    print(f'download automatico indisponivel ({type(erro).__name__}); pegue os arquivos na pasta acima.')
